# insurance-distributional

**Distributional GBM for insurance pricing: model the full conditional distribution, not just E[Y|x].**

Standard pricing models predict the mean. This library predicts the mean *and* the variance — giving you a coefficient of variation per risk. Two policies priced at £350 can have very different volatility profiles. That difference matters for safety loading, reinsurance attachment, and capital allocation.

Implements the ASTIN 2024 Best Paper approach (So & Valdez, arXiv 2406.16206) with CatBoost.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-distributional/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-distributional catboost polars numpy scipy

## Synthetic motor pure premium data

We generate 1,000 policies with a Tweedie (compound Poisson-Gamma) loss distribution.
Crucially, dispersion varies with vehicle age — older vehicles have more volatile losses.
A model that only estimates the mean cannot differentiate these risks.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n = 1_000

vehicle_age  = rng.integers(0, 15, n).astype(float)
driver_age   = rng.integers(18, 75, n).astype(float)
ncd_years    = rng.integers(0, 5, n).astype(float)
vehicle_group = rng.integers(1, 6, n).astype(float)  # 1=small, 5=prestige

X = np.column_stack([vehicle_age, driver_age, ncd_years, vehicle_group])

# True mean
log_mu = (
    5.0
    + 0.02 * np.maximum(30 - driver_age, 0)
    + 0.05 * vehicle_age
    - 0.10 * ncd_years
    + 0.08 * vehicle_group
)
mu_true  = np.exp(log_mu)
phi_true = 0.5 + 0.02 * vehicle_age  # older vehicles -> more dispersed

# Simulate Tweedie outcomes (compound Poisson-Gamma, p=1.5)
p_tweedie  = 1.5
alpha_sev  = (2 - p_tweedie) / (p_tweedie - 1)  # = 1.0
lam        = mu_true ** (2 - p_tweedie) / (phi_true * (2 - p_tweedie))
counts     = rng.poisson(lam)
beta_sev   = mu_true / (lam * alpha_sev)
y = np.array([
    rng.gamma(alpha_sev, beta_sev[i], size=c).sum() if c > 0 else 0.0
    for i, c in enumerate(counts)
])
exposure = rng.uniform(0.5, 1.0, n)

n_train = int(0.8 * n)
X_train, X_test       = X[:n_train], X[n_train:]
y_train, y_test       = y[:n_train], y[n_train:]
exp_train, exp_test   = exposure[:n_train], exposure[n_train:]
phi_test              = phi_true[n_train:]

print(f"Training: {n_train} policies | Test: {n - n_train} policies")
print(f"Zero-claim rate: {(y == 0).mean():.1%}")
print(f"True phi range: {phi_true.min():.2f} – {phi_true.max():.2f} (varies with vehicle age)")

## Fit TweedieGBM

`TweedieGBM` fits two CatBoost models: one for the mean (mu) using Tweedie loss, and one for the dispersion (phi) using squared Pearson residuals via the Smyth-Jorgensen double GLM approach. K-fold cross-fitting prevents the phi model from seeing in-sample mu residuals, which would cause severe undercoverage.

In [ ]:
from insurance_distributional import TweedieGBM

model = TweedieGBM(power=1.5)
model.fit(X_train, y_train, exposure=exp_train)

pred = model.predict(X_test, exposure=exp_test)

print("Per-policy predictions (first 5):")
for i in range(5):
    print(f"  Policy {i}: mean=£{pred.mean[i]:.0f}, CoV={pred.cov[i]:.2f}, "
          f"true_phi={phi_test[i]:.2f}")

## Coverage calibration

The critical test for a distributional model: do the 80%/90%/95% prediction intervals contain the right fraction of actual outcomes? A constant-phi model fails this because it assumes all risks have the same variance.

In [ ]:
from insurance_distributional import coverage

cov_levels = coverage(y_test, pred, levels=(0.80, 0.90, 0.95))

print("Coverage calibration:")
print(f"  Target 80%: achieved {cov_levels[0.80]:.1%}")
print(f"  Target 90%: achieved {cov_levels[0.90]:.1%}")
print(f"  Target 95%: achieved {cov_levels[0.95]:.1%}")

import numpy as np
phi_corr = np.corrcoef(pred.phi, phi_test)[0, 1]
print(f"\nPhi correlation with true phi: {phi_corr:.3f}")
print("(A constant-phi GLM would score 0.000 here)")

## Volatility score for safety loading

`volatility_score()` returns the coefficient of variation per risk: CoV = SD/mean. This is the key output beyond the mean. UK actuaries use it to set safety loadings: `P = mu * (1 + k * CoV)`, where k is the firm's risk appetite parameter.

Here we show that the model correctly ranks high-CoV risks (old vehicles) above low-CoV risks (new vehicles), even if the absolute scale needs cross-validation.

In [ ]:
import polars as pl

results_df = pl.DataFrame({
    "vehicle_age": X_test[:, 0].astype(int).tolist(),
    "pred_mean":   pred.mean.tolist(),
    "volatility":  pred.volatility_score().tolist(),
    "true_phi":    phi_test.tolist(),
})

# Average volatility by vehicle age quartile
summary = (
    results_df
    .with_columns([
        pl.when(pl.col("vehicle_age") <= 4).then(pl.lit("0-4 yrs (new)"))
          .when(pl.col("vehicle_age") <= 9).then(pl.lit("5-9 yrs"))
          .otherwise(pl.lit("10+ yrs (old)")).alias("age_band")
    ])
    .group_by("age_band")
    .agg([
        pl.col("volatility").mean().round(3).alias("mean_volatility"),
        pl.col("true_phi").mean().round(3).alias("true_phi"),
        pl.len().alias("n"),
    ])
    .sort("age_band")
)
print(summary)

## What you should see

- Coverage at 80%/90%/95% should be close to nominal (within 2–4pp). A constant-phi GLM would show systematic undercoverage.
- Phi correlation with true phi should be positive (typically +0.5 to +0.8), confirming the model correctly ranks which risks are more volatile.
- Volatility scores should increase with vehicle age — the pattern the model was trained to recover.

The absolute scale of phi has an upward bias of ~5–12% on this DGP. Use `volatility_score()` for relative comparisons and underwriter referral thresholds; cross-validate the absolute value before plugging into safety loading formulas.

## Next steps

- **`ZIPGBM`** — Zero-Inflated Poisson for pet/travel frequency with structural zeros
- **`NegBinomialGBM`** — overdispersed fleet motor frequency
- **`GammaGBM`** — severity-only model with per-risk CoV
- **`model.log_score()`, `model.crps()`** — proper scoring rules for model comparison

**GitHub:** https://github.com/burning-cost/insurance-distributional  
**PyPI:** https://pypi.org/project/insurance-distributional/